# Strava Activities — Data Preprocessing

This notebook reads the raw activity data exported by the fetch notebook, cleans and
transforms it, engineers analytical features, and exports the result to CSV and Excel.

**Pipeline:**
1. Load `strava_activities.json` into a Pandas DataFrame
2. Clean — flatten nested columns, fix types, audit nulls
3. Transform — convert API units to human-readable values
4. Feature engineering — temporal, performance, and categorical features
5. Export to CSV and Excel

### Section 1 — Libraries & Packages

All imports are declared here so every dependency is visible at a glance.

| Group | Packages | Purpose |
|---|---|---|
| Standard library | `pathlib`, `json` | File I/O and path handling |
| External | `pandas` | DataFrame operations and export |
| External | `numpy` | Vectorised operations and NaN handling |

In [16]:
# Standard Library
from pathlib import Path
import json

# External Packages
import pandas as pd
import numpy as np
import openpyxl

### Section 2 — Project Configuration

Defines the paths used throughout the notebook. The source JSON is produced by the
fetch notebook; the CSV and Excel paths are the export targets of this notebook.

| Constant | Description |
|---|---|
| `BASE_DIR` | Project root (current working directory) |
| `OUTPUT_DIR` | Directory for all exported files |
| `JSON_PATH` | Source file — raw activities from the fetch notebook |
| `CSV_PATH` | Export target — cleaned activities as CSV |
| `EXCEL_PATH` | Export target — cleaned activities as Excel |

In [2]:
# Project root — all paths derived from here.
BASE_DIR = Path.cwd()

# ── Directory & file paths ────────────────────────────────────────────────────
OUTPUT_DIR = BASE_DIR / "output"

JSON_PATH  = OUTPUT_DIR / "strava_activities.json"  # Source (fetch notebook output)
CSV_PATH   = OUTPUT_DIR / "strava_activities.csv"   # Export target
EXCEL_PATH = OUTPUT_DIR / "strava_activities.xlsx"  # Export target

### Section 3 — Load JSON into DataFrame

`load_activities()` reads the JSON file and returns a raw Pandas DataFrame.
Each element of the top-level JSON array becomes one row (one activity).

In [3]:
def load_activities(path=JSON_PATH):
    """
    Load the Strava activities JSON file into a Pandas DataFrame.

    Reads the file produced by ``export_to_json()`` in the fetch notebook.
    Each element of the top-level array becomes one row.

    Parameters
    ----------
    path : pathlib.Path or str, optional
        Path to the JSON file. Defaults to ``JSON_PATH``
        (``<project_root>/output/strava_activities.json``).

    Returns
    -------
    pandas.DataFrame
        Raw activities DataFrame as returned by the Strava API.

    Raises
    ------
    FileNotFoundError
        If the JSON file does not exist at ``path``.

    Usage
    -----
    df = load_activities()
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"{path} not found. Run the fetch notebook first to generate it."
        )
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    df = pd.DataFrame(data)
    print(f"Loaded {len(df)} activities with {df.shape[1]} columns.")
    return df

In [4]:
df = load_activities()
df.head(2)

Loaded 48 activities with 52 columns.


,resource_state,athlete,name,distance,moving_time,elapsed_time,total_elevation_gain,type,sport_type,id,...,upload_id,upload_id_str,external_id,from_accepted_tag,pr_count,total_photo_count,has_kudoed,suffer_score,workout_type,device_name
0,2,"{'id': 843017438, 'resource_state': 1}",Outdoor walk,10263.4,6036,6160,79.4,Walk,Walk,18765234837,...,19875070186,19875070186,1780462662492.fit,False,0,0,False,22.0,None,None
1,2,"{'id': 843017438, 'resource_state': 1}",Outdoor walk,10277.0,6087,6395,79.6,Walk,Walk,18750878456,...,19860482229,19860482229,1780376196575.fit,False,0,0,False,23.0,None,None


### Section 4 — Data Cleaning

Three focused functions applied in sequence. **No columns are dropped at this stage** —
all columns are preserved after flattening; column removal will be decided in a later
review pass after inspecting the full flattened structure.

1. `flatten_nested_columns(df)` — unpack dicts and lists into flat scalar columns
2. `parse_and_fix_types(df)` — parse datetimes, fix timezone strings, correct numeric types
3. `handle_nulls(df)` — audit and document null values

#### `flatten_nested_columns(df)` — Flatten Nested API Objects

The Strava API returns `athlete`, `map`, `start_latlng`, and `end_latlng` as nested
objects. This function extracts the useful scalar fields from each and drops the
original nested columns.

In [5]:
def flatten_nested_columns(df):
    """
    Unpack nested dict and list columns into flat scalar columns.

    The Strava API returns several columns as nested objects stored as dicts or lists.
    This function extracts the useful scalar fields and drops the original nested columns,
    making the DataFrame fully flat and ready for type parsing and analysis.

    Transformations applied:
    - ``athlete``      dict → ``athlete_id``  (int)
    - ``map``          dict → ``has_route``   (bool: summary_polyline is non-empty)
    - ``start_latlng`` list → ``start_lat``,  ``start_lng`` (None for indoor activities)
    - ``end_latlng``   list → ``end_lat``,    ``end_lng``   (None for indoor activities)

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame containing raw nested columns from the Strava API.

    Returns
    -------
    pandas.DataFrame
        DataFrame with nested columns replaced by flat scalar equivalents.

    Usage
    -----
    df = flatten_nested_columns(df)
    """
    df = df.copy()

    # athlete dict → athlete_id (int)
    df["athlete_id"] = df["athlete"].apply(
        lambda x: x.get("id") if isinstance(x, dict) else None
    )
    df = df.drop(columns=["athlete"])

    # map dict → has_route (bool: True when a GPS polyline exists)
    df["has_route"] = df["map"].apply(
        lambda x: bool(x.get("summary_polyline")) if isinstance(x, dict) else False
    )
    df = df.drop(columns=["map"])

    # start_latlng list → start_lat, start_lng (None for empty lists / indoor activities)
    df["start_lat"] = df["start_latlng"].apply(
        lambda x: x[0] if isinstance(x, list) and len(x) == 2 else None
    )
    df["start_lng"] = df["start_latlng"].apply(
        lambda x: x[1] if isinstance(x, list) and len(x) == 2 else None
    )
    df = df.drop(columns=["start_latlng"])

    # end_latlng list → end_lat, end_lng
    df["end_lat"] = df["end_latlng"].apply(
        lambda x: x[0] if isinstance(x, list) and len(x) == 2 else None
    )
    df["end_lng"] = df["end_latlng"].apply(
        lambda x: x[1] if isinstance(x, list) and len(x) == 2 else None
    )
    df = df.drop(columns=["end_latlng"])

    print(f"After flattening: {df.shape[1]} columns.")
    return df

#### `parse_and_fix_types(df)` — Type Parsing & Corrections

Parses ISO 8601 datetime strings into timezone-aware Timestamps, converts
`utc_offset` from float seconds to integer hours, and strips escaped forward
slashes from timezone name strings.

In [6]:
def parse_and_fix_types(df):
    """
    Parse datetime columns, fix timezone strings, and correct numeric types.

    Transformations applied:
    - ``start_date`` and ``start_date_local``: ISO 8601 string → timezone-aware datetime (UTC)
    - ``timezone``: strip escaped forward slashes left by some JSON serialisers
      (e.g. "Africa\/Addis_Ababa" → "Africa/Addis_Ababa")
    - ``utc_offset``: float seconds → int hours (10800.0 → 3)

    Parameters
    ----------
    df : pandas.DataFrame

    Returns
    -------
    pandas.DataFrame

    Usage
    -----
    df = parse_and_fix_types(df)
    """
    df = df.copy()

    # Parse ISO 8601 datetime strings to tz-aware Timestamps (UTC)
    df["start_date"]       = pd.to_datetime(df["start_date"],       utc=True)
    df["start_date_local"] = pd.to_datetime(df["start_date_local"], utc=True)

    # Strip escaped forward slashes (safety: json.load handles this, but some serialisers miss it)
    df["timezone"] = df["timezone"].str.replace("\\/", "/", regex=False)

    # Convert utc_offset from float seconds to integer hours
    df["utc_offset"] = (df["utc_offset"] / 3600).astype(int)

    return df

#### `handle_nulls(df)` — Null Audit Checkpoint

Prints a summary of all null values. No imputation is applied — every null reflects
a genuine absence of data (e.g. indoor activities have no GPS or cadence data).

In [7]:
def handle_nulls(df):
    """
    Audit null values and document the expected patterns. No imputation is applied.

    Null audit results:
    - ``average_heartrate``, ``max_heartrate``, ``suffer_score``:
      NaN for activities recorded without a heart rate sensor (2 of 46).
    - ``average_cadence``:
      NaN for indoor (trainer) workouts. Expected absence — no footpod or pedal sensor.
    - ``start_lat``, ``start_lng``, ``end_lat``, ``end_lng``:
      None for indoor activities. No GPS data by design.
    - ``location_city``, ``location_state``, ``location_country``:
      All null. Retained for the column review pass.

    Parameters
    ----------
    df : pandas.DataFrame

    Returns
    -------
    pandas.DataFrame
        Unchanged (this function is a documented audit checkpoint).

    Usage
    -----
    df = handle_nulls(df)
    """
    df = df.copy()

    null_summary = df.isnull().sum()
    null_cols    = null_summary[null_summary > 0]

    print("Null value summary:")
    print(null_cols.to_string())

    return df

### Section 5 — Unit Conversions

The Strava API returns distances in metres, speeds in m/s, and times in seconds.
`convert_units()` creates human-readable equivalents and drops the raw originals.

| New column | Source | Formula |
|---|---|---|
| `distance_km` | `distance` (m) | ÷ 1000 |
| `moving_time_min` | `moving_time` (s) | ÷ 60 |
| `elapsed_time_min` | `elapsed_time` (s) | ÷ 60 |
| `average_speed_kmh` | `average_speed` (m/s) | × 3.6 |
| `max_speed_kmh` | `max_speed` (m/s) | × 3.6 |

Elevation columns (`elev_high`, `elev_low`, `total_elevation_gain`) remain in metres.

In [8]:
def convert_units(df):
    """
    Convert Strava API raw units to human-readable equivalents.

    Creates new columns with familiar units, rounds them to a sensible precision,
    and drops the raw API originals.

    Conversions applied:
    - ``distance``      (m)   → ``distance_km``       (km)   = distance / 1000
    - ``moving_time``   (sec) → ``moving_time_min``   (min)  = moving_time / 60
    - ``elapsed_time``  (sec) → ``elapsed_time_min``  (min)  = elapsed_time / 60
    - ``average_speed`` (m/s) → ``average_speed_kmh`` (km/h) = average_speed × 3.6
    - ``max_speed``     (m/s) → ``max_speed_kmh``     (km/h) = max_speed × 3.6

    Elevation columns (``elev_high``, ``elev_low``, ``total_elevation_gain``) are left
    in metres as that is the standard unit for elevation.

    Parameters
    ----------
    df : pandas.DataFrame

    Returns
    -------
    pandas.DataFrame

    Usage
    -----
    df = convert_units(df)
    """
    df = df.copy()

    df["distance_km"]       = (df["distance"]      / 1000).round(3)
    df["moving_time_min"]   = (df["moving_time"]   / 60  ).round(2)
    df["elapsed_time_min"]  = (df["elapsed_time"]  / 60  ).round(2)
    df["average_speed_kmh"] = (df["average_speed"] * 3.6 ).round(3)
    df["max_speed_kmh"]     = (df["max_speed"]     * 3.6 ).round(3)

    df = df.drop(columns=["distance", "moving_time", "elapsed_time", "average_speed", "max_speed"])

    return df

### Section 6 — Feature Engineering

`engineer_features()` derives three groups of new analytical columns:

**Temporal** (from `start_date_local`): `date`, `year`, `month`, `month_name`, `day_of_week` (0=Sat), `day_name`, `hour_of_day`, `week_of_year`, `is_weekend`, `time_of_day`

**Performance**: `pace_min_per_km` (NaN for indoor), `is_outdoor`

**Categorical**: `distance_bucket` (Indoor / Short / Medium / Long), `hr_zone` (Zones 1–5 based on max HR 182 bpm)

In [9]:
def engineer_features(df):
    """
    Derive analytical features from existing columns.

    Adds three groups of new columns without modifying the originals:

    **Temporal features** (derived from ``start_date_local``):
    - ``date``           — Date only (no time component)
    - ``year``           — Calendar year
    - ``month``          — Month number (1–12)
    - ``month_name``     — Month name (e.g. "January")
    - ``day_of_week``    — Day index: Saturday = 0, Sunday = 1, ..., Friday = 6
    - ``day_name``       — Day name (e.g. "Monday")
    - ``hour_of_day``    — Hour of the day (0–23)
    - ``week_of_year``   — ISO week number (1–53)
    - ``is_weekend``     — True for Saturday and Sunday
    - ``time_of_day``    — "Morning" (5–11h), "Afternoon" (12–17h),
                           "Evening" (18–21h), "Night" (22–4h)

    **Performance features**:
    - ``pace_min_per_km`` — Minutes per km. NaN for indoor activities (distance = 0 km).
    - ``is_outdoor``      — True when not a trainer workout and GPS coordinates exist.

    **Categorisation**:
    - ``distance_bucket`` — "Indoor" (0 km) | "Short" (<5 km) |
                            "Medium" (5–15 km) | "Long" (>15 km)
    - ``hr_zone``         — Heart rate zone based on max HR of 182 bpm. NaN if no HR data.
                            Zone 1: <109 | Zone 2: 109–127 | Zone 3: 128–147 |
                            Zone 4: 148–163 | Zone 5: ≥164

    Parameters
    ----------
    df : pandas.DataFrame
        Must contain ``start_date_local``, ``moving_time_min``, ``distance_km``,
        ``trainer``, ``start_lat``, and ``average_heartrate``.

    Returns
    -------
    pandas.DataFrame

    Usage
    -----
    df = engineer_features(df)
    """
    df = df.copy()
    dt = df["start_date_local"].dt

    # ── Temporal features ─────────────────────────────────────────────────────
    df["date"]         = dt.date
    df["year"]         = dt.year
    df["month"]        = dt.month
    df["month_name"]   = dt.month_name()
    # Shift so Saturday = 0 (pandas default: Monday = 0, Saturday = 5)
    df["day_of_week"]  = (dt.dayofweek + 2) % 7
    df["day_name"]     = dt.day_name()
    df["hour_of_day"]  = dt.hour
    df["week_of_year"] = dt.isocalendar().week.astype(int)
    df["is_weekend"]   = df["day_of_week"].isin([0, 1])  # Saturday = 0, Sunday = 1

    def _time_of_day(hour):
        if 5 <= hour <= 11:
            return "Morning"
        elif 12 <= hour <= 17:
            return "Afternoon"
        elif 18 <= hour <= 21:
            return "Evening"
        else:
            return "Night"

    df["time_of_day"] = df["hour_of_day"].apply(_time_of_day)

    # ── Performance features ──────────────────────────────────────────────────
    df["pace_min_per_km"] = np.where(
        df["distance_km"] > 0,
        (df["moving_time_min"] / df["distance_km"]).round(2),
        np.nan
    )
    df["is_outdoor"] = ~df["trainer"] & df["start_lat"].notna()

    # ── Distance categorisation ───────────────────────────────────────────────
    conditions = [
        df["distance_km"] == 0,
        df["distance_km"] < 5,
        df["distance_km"] < 15,
    ]
    choices = ["Indoor", "Short", "Medium"]
    df["distance_bucket"] = np.select(conditions, choices, default="Long")

    # ── Heart rate zone (max HR = 182 bpm) ───────────────────────────────────
    hr_bins   = [0, 109, 128, 148, 164, float("inf")]
    hr_labels = ["Zone 1", "Zone 2", "Zone 3", "Zone 4", "Zone 5"]
    df["hr_zone"] = pd.cut(
        df["average_heartrate"],
        bins=hr_bins,
        labels=hr_labels,
        right=False
    )

    print(f"Feature engineering complete. DataFrame now has {df.shape[1]} columns.")
    return df

### Section 7 — Preview Cleaned DataFrame

Run the full pipeline up to this point and inspect the result before exporting.

In [10]:
print(f'Shape: {df.shape}')
df.info()

Shape: (48, 52)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48 entries, 0 to 47
Data columns (total 52 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   resource_state                 48 non-null     int64  
 1   athlete                        48 non-null     object 
 2   name                           48 non-null     object 
 3   distance                       48 non-null     float64
 4   moving_time                    48 non-null     int64  
 5   elapsed_time                   48 non-null     int64  
 6   total_elevation_gain           48 non-null     float64
 7   type                           48 non-null     object 
 8   sport_type                     48 non-null     object 
 9   id                             48 non-null     int64  
 10  start_date                     48 non-null     object 
 11  start_date_local               48 non-null     object 
 12  timezone                       48 no

In [11]:
df.describe(include='all')

,resource_state,athlete,name,distance,moving_time,elapsed_time,total_elevation_gain,type,sport_type,id,...,upload_id,upload_id_str,external_id,from_accepted_tag,pr_count,total_photo_count,has_kudoed,suffer_score,workout_type,device_name
count,48.0,48,48,48.000000,48.000000,48.000000,48.000000,48,48,4.800000e+01,...,4.800000e+01,48,48,48,48.0,48.0,48,46.000000,0,1
unique,NaN,1,4,NaN,NaN,NaN,NaN,3,3,NaN,...,NaN,48,48,1,NaN,NaN,1,NaN,0,1
top,NaN,"{'id': 843017438, 'resource_state': 1}",Outdoor walk,NaN,NaN,NaN,NaN,Walk,Walk,NaN,...,NaN,19875070186,1780462662492.fit,False,NaN,NaN,False,NaN,NaN,Strava App
freq,NaN,48,37,NaN,NaN,NaN,NaN,38,38,NaN,...,NaN,1,1,48,NaN,NaN,48,NaN,NaN,1
mean,2.0,NaN,NaN,6387.691667,4195.854167,4391.562500,50.000000,NaN,NaN,1.839093e+10,...,1.949551e+10,NaN,NaN,NaN,0.0,0.0,NaN,16.130435,NaN,NaN
std,0.0,NaN,NaN,4226.055403,2412.073938,2493.909013,32.778489,NaN,NaN,1.880231e+08,...,1.900078e+08,NaN,NaN,NaN,0.0,0.0,NaN,7.747713,NaN,NaN
min,2.0,NaN,NaN,0.000000,100.000000,100.000000,0.000000,NaN,NaN,1.808745e+10,...,1.918919e+10,NaN,NaN,NaN,0.0,0.0,NaN,0.000000,NaN,NaN
25%,2.0,NaN,NaN,2815.850000,1871.500000,2010.250000,28.600000,NaN,NaN,1.824154e+10,...,1.934447e+10,NaN,NaN,NaN,0.0,0.0,NaN,11.250000,NaN,NaN
50%,2.0,NaN,NaN,7630.750000,4980.500000,5343.500000,55.200000,NaN,NaN,1.840412e+10,...,1.950867e+10,NaN,NaN,NaN,0.0,0.0,NaN,17.500000,NaN,NaN
75%,2.0,NaN,NaN,10261.375000,6398.500000,6572.250000,79.450000,NaN,NaN,1.854084e+10,...,1.964675e+10,NaN,NaN,NaN,0.0,0.0,NaN,22.000000,NaN,NaN


In [12]:
df.head()

,resource_state,athlete,name,distance,moving_time,elapsed_time,total_elevation_gain,type,sport_type,id,...,upload_id,upload_id_str,external_id,from_accepted_tag,pr_count,total_photo_count,has_kudoed,suffer_score,workout_type,device_name
0,2,"{'id': 843017438, 'resource_state': 1}",Outdoor walk,10263.4,6036,6160,79.4,Walk,Walk,18765234837,...,19875070186,19875070186,1780462662492.fit,False,0,0,False,22.0,None,None
1,2,"{'id': 843017438, 'resource_state': 1}",Outdoor walk,10277.0,6087,6395,79.6,Walk,Walk,18750878456,...,19860482229,19860482229,1780376196575.fit,False,0,0,False,23.0,None,None
2,2,"{'id': 843017438, 'resource_state': 1}",HIIT,0.0,1005,1005,0.0,Workout,HighIntensityIntervalTraining,18629572376,...,19736833077,19736833077,1779601140871.fit,False,0,0,False,15.0,None,None
3,2,"{'id': 843017438, 'resource_state': 1}",Outdoor walk,12095.7,7048,7200,104.0,Walk,Walk,18629572464,...,19736833135,19736833135,1779601141612.fit,False,0,0,False,25.0,None,None
4,2,"{'id': 843017438, 'resource_state': 1}",HIIT,0.0,100,100,0.0,Workout,HighIntensityIntervalTraining,18616181767,...,19723147664,19723147664,1779514406022.fit,False,0,0,False,1.0,None,None


### Section 8 — Export

Two functions that write the preprocessed DataFrame to disk:
- `export_to_csv()` — CSV with UTF-8 BOM so Excel opens it without encoding issues
- `export_to_excel()` — Excel workbook via `openpyxl`

In [13]:
def export_to_csv(df, path=CSV_PATH):
    """
    Export the activities DataFrame to a CSV file.

    Uses ``utf-8-sig`` encoding (UTF-8 with BOM) so Excel opens the file correctly
    without character encoding issues on any platform.

    Parameters
    ----------
    df : pandas.DataFrame
        The preprocessed activities DataFrame.
    path : pathlib.Path or str, optional
        Destination file path. Defaults to ``CSV_PATH``
        (``<project_root>/output/strava_activities.csv``).

    Returns
    -------
    None
        Writes the file and prints a confirmation message.

    Usage
    -----
    export_to_csv(df)
    export_to_csv(df, path="my_folder/activities.csv")
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=False, encoding="utf-8-sig")
    print(f"Exported {len(df)} activities to {path}")

In [14]:
def export_to_excel(df, path=EXCEL_PATH):
    """
    Export the activities DataFrame to an Excel workbook.

    Writes a single worksheet named "Strava Activities" using the openpyxl engine.
    Timezone-aware datetime columns are converted to timezone-naive before writing
    because Excel does not support timezone-aware timestamps.

    Parameters
    ----------
    df : pandas.DataFrame
        The preprocessed activities DataFrame.
    path : pathlib.Path or str, optional
        Destination file path. Defaults to ``EXCEL_PATH``
        (``<project_root>/output/strava_activities.xlsx``).

    Returns
    -------
    None
        Writes the file and prints a confirmation message.

    Usage
    -----
    export_to_excel(df)
    export_to_excel(df, path="my_folder/activities.xlsx")
    """
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    # Excel does not support tz-aware datetimes — strip timezone info before writing.
    df_excel = df.copy()
    for col in df_excel.select_dtypes(include=["datetimetz"]).columns:
        df_excel[col] = df_excel[col].dt.tz_localize(None)

    df_excel.to_excel(path, index=False, sheet_name="Strava Activities", engine="openpyxl")
    print(f"Exported {len(df)} activities to {path}")

### Pipeline Orchestration

`run_pipeline()` chains all steps in order and returns the final DataFrame.
Run this single cell to execute the complete preprocessing workflow end-to-end.

In [17]:
def run_pipeline():
    """
    Run the full preprocessing pipeline end-to-end.

    Chains all preprocessing steps in order:
    1. Load JSON → DataFrame
    2. Flatten nested columns
    3. Parse types & fix strings
    4. Audit nulls
    5. Convert units
    6. Engineer features
    7. Export to CSV and Excel

    Returns
    -------
    pandas.DataFrame
        Fully cleaned, transformed, and feature-engineered activities DataFrame.

    Usage
    -----
    df = run_pipeline()
    """
    df = load_activities()
    df = flatten_nested_columns(df)
    df = parse_and_fix_types(df)
    df = handle_nulls(df)
    df = convert_units(df)
    df = engineer_features(df)
    export_to_csv(df)
    export_to_excel(df)
    return df


df = run_pipeline()
df

Loaded 48 activities with 52 columns.
After flattening: 54 columns.
Null value summary:
location_city        48
location_state       48
location_country     48
gear_id              48
average_cadence      10
average_heartrate     2
max_heartrate         2
suffer_score          2
workout_type         48
device_name          47
start_lat             9
start_lng             9
end_lat               9
end_lng               9
Feature engineering complete. DataFrame now has 68 columns.
Exported 48 activities to /Users/waleedmouhammed/Documents/DataAnalysisWorkshop/StravaDataAnalytics/Strava_Performance_Analysis/output/strava_activities.csv
Exported 48 activities to /Users/waleedmouhammed/Documents/DataAnalysisWorkshop/StravaDataAnalytics/Strava_Performance_Analysis/output/strava_activities.xlsx


,resource_state,name,total_elevation_gain,type,sport_type,id,start_date,start_date_local,timezone,utc_offset,...,day_of_week,day_name,hour_of_day,week_of_year,is_weekend,time_of_day,pace_min_per_km,is_outdoor,distance_bucket,hr_zone
0,2,Outdoor walk,79.4,Walk,Walk,18765234837,2026-06-03 03:12:45+00:00,2026-06-03 06:12:45+00:00,(GMT+02:00) Africa/Cairo,3,...,4,Wednesday,6,23,False,Morning,9.80,True,Medium,Zone 2
1,2,Outdoor walk,79.6,Walk,Walk,18750878456,2026-06-02 03:07:48+00:00,2026-06-02 06:07:48+00:00,(GMT+02:00) Africa/Cairo,3,...,3,Tuesday,6,23,False,Morning,9.87,True,Medium,Zone 2
2,2,HIIT,0.0,Workout,HighIntensityIntervalTraining,18629572376,2026-05-24 05:15:45+00:00,2026-05-24 08:15:45+00:00,(GMT+03:00) Africa/Addis_Ababa,3,...,1,Sunday,8,21,True,Morning,NaN,False,Indoor,Zone 3
3,2,Outdoor walk,104.0,Walk,Walk,18629572464,2026-05-24 02:58:56+00:00,2026-05-24 05:58:56+00:00,(GMT+02:00) Africa/Cairo,3,...,1,Sunday,5,21,True,Morning,9.71,True,Medium,Zone 2
4,2,HIIT,0.0,Workout,HighIntensityIntervalTraining,18616181767,2026-05-23 05:23:08+00:00,2026-05-23 08:23:08+00:00,(GMT+03:00) Africa/Addis_Ababa,3,...,0,Saturday,8,21,True,Morning,NaN,False,Indoor,Zone 3
5,2,Outdoor walk,80.1,Walk,Walk,18616181895,2026-05-23 03:14:14+00:00,2026-05-23 06:14:14+00:00,(GMT+02:00) Africa/Cairo,3,...,0,Saturday,6,21,True,Morning,10.30,True,Medium,Zone 2
6,2,HIIT,0.0,Workout,HighIntensityIntervalTraining,18604854789,2026-05-22 06:58:16+00:00,2026-05-22 09:58:16+00:00,(GMT+03:00) Africa/Addis_Ababa,3,...,6,Friday,9,21,False,Morning,NaN,False,Indoor,Zone 3
7,2,Outdoor walk,51.7,Walk,Walk,18597799203,2026-05-21 15:38:39+00:00,2026-05-21 18:38:39+00:00,(GMT+02:00) Africa/Cairo,3,...,5,Thursday,18,21,False,Evening,9.80,True,Medium,Zone 2
8,2,HIIT,0.0,Workout,HighIntensityIntervalTraining,18596369476,2026-05-19 04:29:35+00:00,2026-05-19 07:29:35+00:00,(GMT+03:00) Africa/Addis_Ababa,3,...,3,Tuesday,7,21,False,Morning,NaN,False,Indoor,Zone 3
9,2,Outdoor walk,57.5,Walk,Walk,18596369726,2026-05-19 03:04:58+00:00,2026-05-19 06:04:58+00:00,(GMT+02:00) Africa/Cairo,3,...,3,Tuesday,6,21,False,Morning,10.26,True,Medium,Zone 2
